# ColumnAnonymizer Module Demo

This notebook demonstrates how to use the `ColumnAnonymizer` class from `pat2vec.util.anonymize_column`.

## Overview

The `ColumnAnonymizer` provides:
- **Deterministic pseudonymisation** of sensitive data using keyed HMAC hashes
- **Consistent mappings** across sessions by saving/loading mappings
- Support for custom column names (defaults to 'client_idcode')
- Data integrity while hiding sensitive values

## Key Features

1. **Pseudonymize**: Convert original values to deterministic hash values
2. **De-pseudonymize**: Reveal original values using the stored mapping
3. **Persistence**: Save/load mappings for consistency across sessions
4. **Convenience functions**: Simple wrapper functions for one-off operations

In [ ]:
import sys

sys.path.insert(0, "../..")

import pandas as pd
from pat2vec.util.anonymize_column import (
    ColumnAnonymizer,
    anonymize_column,
    deanonymize_column,
)

---

## Quick Start Example

In [ ]:
# Create sample data similar to what you'd see in pat2vec
df = pd.DataFrame(
    {
        "client_idcode": ["A001", "B002", "C003", "D004", "E005"],
        "patient_name": [
            "John Doe",
            "Jane Smith",
            "Bob Johnson",
            "Alice Brown",
            "Charlie Wilson",
        ],
        "age": [45, 52, 67, 38, 59],
    }
)

print("Original DataFrame:")
df

## Method 1: Using the ColumnAnonymizer Class

### Step 1: Initialize with a secret key

In [ ]:
# Create anonymizer instance with a secret key
# The same key will produce consistent hashes across runs
anonymizer = ColumnAnonymizer(key="my_secret_anonymization_key_2024")

print(f"Using key hash: {hash(anonymizer.key)}")

### Step 2: Anonymize the data

In [ ]:
# Anonymize the client_idcode column
df_anon, mapping = anonymizer.anonymize(df, column_name="client_idcode")

print("Anonymized DataFrame:")
df_anon

print("\nMapping (keep this secret!):")
for orig, anon in list(mapping.items())[:3]:
    print(f"  {orig} → {anon}")

### Step 3: Save the mapping for later use

In [ ]:
# Save the anonymization key to a file
import os

output_dir = "anonymization_demo"
os.makedirs(output_dir, exist_ok=True)

key_path = os.path.join(output_dir, "anonymization_key.json")
anonymizer.save_mapping(mapping, key_path)

print(f"Key saved to: {key_path}")

# Show the saved content
with open(key_path, "r") as f:
    import json

    print("\nSaved key content:")
    print(json.dumps(json.load(f), indent=2))

### Step 4: De-anonymize using the mapping

In [ ]:
# De-anonymize back to original values
df_deanon = anonymizer.deanonymize(df_anon, mapping, column_name="client_idcode")

print("De-anonymized DataFrame:")
df_deanon

print(
    "\nVerification - Original equals de-anonymized:",
    df["client_idcode"].equals(df_deanon["client_idcode"]),
)

## Method 2: Using Convenience Functions

For simple one-off operations without needing to manage class instances.

In [ ]:
import os

# Create fresh sample data
df = pd.DataFrame({"client_idcode": ["P001", "P002", "P003"], "value": [100, 200, 300]})

print("Original data:")
df

# Use convenience function to anonymize
df_anon, mapping = anonymize_column(
    df, column_name="client_idcode", key="simple_key_123"
)

print("\nAnonymized with convenience function:")
df_anon

# De-anonymize
df_restored = deanonymize_column(df_anon, mapping, column_name="client_idcode")

print("\nRestored data:")
df_restored

## Loading Saved Keys for Consistency Across Sessions

This is crucial when you need the same anonymization in multiple runs or sessions.

In [ ]:
# Start fresh - simulate a new session/load from disk
from pat2vec.util.anonymize_column import ColumnAnonymizer

# Load the previously saved mapping
loaded_anonymizer = ColumnAnonymizer(key="my_secret_anonymization_key_2024")

key_path = "anonymization_demo/anonymization_key.json"
loaded_mapping = loaded_anonymizer.load_mapping(key_path)

print("Loaded mapping:")
for orig, anon in list(loaded_mapping.items())[:3]:
    print(f"  {orig} → {anon}")

# Use the loaded mapping on the same data
df_new_session = pd.DataFrame(
    {"client_idcode": ["A001", "B002", "C003"], "data": [1, 2, 3]}
)

df_anon_session, _ = loaded_anonymizer.anonymize(
    df_new_session, column_name="client_idcode"
)

print("\nNew session anonymized data:")
df_anon_session

## Error Handling

### Case 1: Column doesn't exist

In [ ]:
from pat2vec.util.anonymize_column import ColumnAnonymizer

anonymizer = ColumnAnonymizer(key="test_key")

try:
    df_test = pd.DataFrame({"wrong_column": ["A", "B"]})
    anonymizer.anonymize(df_test, column_name="nonexistent")
except KeyError as e:
    print(f"✓ Caught expected error:\n{e}")

### Case 2: Missing hash in mapping during de-anonymization

In [ ]:
from pat2vec.util.anonymize_column import ColumnAnonymizer

anonymizer = ColumnAnonymizer(key="test_key")

# Create data with one value not in the original mapping
df_original = pd.DataFrame({"client_idcode": ["A", "B", "C"]})
df_anon, mapping = anonymizer.anonymize(df_original)

print("Original mapping:")
for k, v in mapping.items():
    print(f"  {k} → {v}")

# Try to de-anonymize with a different value not in the original set
df_new = pd.DataFrame({"client_idcode": ["A", "D"]})
print("\nAttempting to de-anonymize values: ['A', 'D']")
try:
    df_deanon = anonymizer.deanonymize(df_new, mapping)
    print("Result:")
    df_deanon
except KeyError as e:
    print(f"KeyError: {e}")

## Performance and Data Types

### Handling Null Values

In [ ]:
import pandas as pd
from pat2vec.util.anonymize_column import ColumnAnonymizer
import numpy as np

# DataFrame with null values
df_nulls = pd.DataFrame(
    {"client_idcode": ["A001", None, "C003", np.nan, "E005"], "value": [1, 2, 3, 4, 5]}
)

print("Original data with nulls:")
df_nulls

anonymizer = ColumnAnonymizer(key="null_test_key")
df_anon_nulls, mapping = anonymizer.anonymize(df_nulls, column_name="client_idcode")

print("\nAnonymized (nulls preserved):")
df_anon_nulls

print(f"\nMapping created for {len(mapping)} unique values (nulls excluded)")

### Large Scale Example

In [ ]:
import pandas as pd
from pat2vec.util.anonymize_column import ColumnAnonymizer
import time

# Create large dataset
n_patients = 10000
df_large = pd.DataFrame(
    {
        "client_idcode": [f"P{i:04d}" for i in range(n_patients)],
        "score": __import__("numpy").random.randn(n_patients),
    }
)

print(f"Working with {len(df_large)} patients")

# Time the pseudonymization
start = time.time()
anonymizer = ColumnAnonymizer(key="large_scale_key")
df_anon, mapping = anonymizer.anonymize(df_large, column_name="client_idcode")
elapsed = time.time() - start

print(f"Pseudonymized {len(df_large)} patients in {elapsed:.3f} seconds")
print(f"Unique values mapped: {len(mapping)}")
print("\nSample of pseudonymized data:")
df_anon.head(10)

## Security Considerations

### Key Management
- Keep your anonymization key **secret** - anyone with the key can de-anonymize your data
- Store keys securely (environment variables, secret management systems)
- Use strong, unique keys for each project/environment

In [ ]:
# Example of using stronger hashing with custom key
import hmac


class CustomAnonymizer(ColumnAnonymizer):
    def _hash_value(self, value: str) -> str:
        """Use HMAC for better cryptographic practice"""
        return hmac.new(
            self.key.encode(), value.encode(), __import__("hashlib").sha256
        ).hexdigest()[:32]


# Usage
custom_anon = CustomAnonymizer(key="strong_crypto_key_2024")
df_test = pd.DataFrame({"client_idcode": ["A", "B", "C"]})
df_custom, _ = custom_anon.anonymize(df_test)
print("Custom HMAC hashing:")
df_custom

## Integration with pat2vec Workflows

The anonymizer can be integrated into your pat2vec data pipeline to protect sensitive identifiers while processing.

In [ ]:
import pandas as pd
from pat2vec.util.anonymize_column import ColumnAnonymizer
import os

# Simulate a pat2vec-style patient cohort file
df_patients = pd.DataFrame(
    {
        "client_idcode": [f"PAT_{i}" for i in range(10)],
        "site": ["A", "B", "A", "C", "B", "A", "C", "A", "B", "C"],
        "admission_date": pd.date_range("2024-01-01", periods=10, freq="D"),
    }
)

print("Original patient cohort:")
df_patients.head()

# Create anonymization key
output_dir = "anonymization_demo"
os.makedirs(output_dir, exist_ok=True)

key_path = os.path.join(output_dir, "patient_cohort_key.json")

# Anonymize and save
anonymizer = ColumnAnonymizer(key="pat2vec_patient_cohort_2024")
df_anon, mapping = anonymizer.anonymize(df_patients, column_name="client_idcode")
anonymizer.save_mapping(mapping, key_path)

print("\nAnonymized cohort (ready for analysis):")
df_anon.head()

# Later, in a separate session or for different analysis...
# Load the saved mapping to maintain consistency
loaded_mapping = anonymizer.load_mapping(key_path)
df_check = pd.DataFrame({"client_idcode": ["PAT_0", "PAT_1"]})
df_check_anon, _ = anonymizer.anonymize(df_check, column_name="client_idcode")

print("\nConsistent pseudonymization across sessions:")
for orig in ["PAT_0", "PAT_1"]:
    print(f"  {orig} → {loaded_mapping[orig]}")

## Summary

The `ColumnAnonymizer` provides a robust, deterministic pseudonymization solution for pat2vec workflows. Key points:

✅ **Deterministic** - Same key = same hash across sessions  
✅ **Reversible** - Can de-pseudonymize using the stored mapping  
✅ **Flexible** - Custom column names, save/load mappings  
✅ **Secure** - Uses HMAC-SHA256 for cryptographic hashing  

For production use:
- Always keep keys secure
- Use securely generated random keys (default behavior)
- Consider longer hash outputs for large datasets